# Operate: production에서 Managed Agents 실행하기

대부분의 다른 Managed Agents cookbook은 agent 루프 자체에 집중하며 fixture를 대상으로 유용한 작업을 수행하게 합니다. 이 cookbook은 그 루프를 둘러싼 장치, 즉 Managed Agents 앱을 실제 사용자에게 제공하기 전에 필요한 요소에 관한 것입니다.

1. agent가 모든 호출을 애플리케이션으로 왕복시키지 않고 SaaS API와 통신해야 할 때 custom tool 대신 사용하는 **MCP toolset**.
2. 최종 사용자별 credential을 보관하는 **Vault**. 각 사용자의 GitHub / Linear / Slack token을 다른 사용자와 분리하고 audit trail을 명확하게 유지합니다.
3. 내내 장시간 HTTP 연결을 열어 두지 않고 human-in-the-loop 작업을 진행하는 **Webhook**.
4. 시간이 지나면서 workspace에 누적되는 항목을 관리하는 **resource lifecycle** 동작(list, retrieve, update, archive, delete).
5. compliance가 model 요청을 특정 지역에서 실행하도록 요구할 때의 **inference 지역 고정**(`inference_geo`).

앞의 네 항목을 다루는 하나의 전체 흐름을 만들고 지역 고정으로 마무리하겠습니다. 가상의 최종 사용자용 vault를 만들고, GitHub MCP credential을 연결하고, 서버 측에서 credential을 사용하는 agent 세션을 실행하고, 실제 production server에서 같은 세션을 진행하기 위해 등록할 webhook handler를 보여 준 다음, 나중에 정리할 때 사용할 관리 동작을 살펴봅니다.

이 notebook은 environment에 `GITHUB_TOKEN`이 필요합니다.


In [ ]:
import os

from anthropic import Anthropic
from utilities import stream_until_end_turn, wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
client = Anthropic()

GH_TOKEN = os.environ.get("GITHUB_TOKEN")
if not GH_TOKEN:
    raise SystemExit("Set GITHUB_TOKEN to run this notebook.")

## 개념: MCP toolset과 vault

**MCP toolset**은 custom tool(gate notebook)과 `resources=` mount(orchestrate notebook)에 이은 세 번째 확장 패턴입니다. MCP toolset은 agent가 [Model Context Protocol](https://modelcontextprotocol.io/)을 구현하는 외부 server를 사용하게 합니다. agent는 애플리케이션을 왕복하지 않고 sandbox 안에서 해당 server의 tool을 직접 호출합니다. Anthropic이 호출을 proxy하고 server가 응답하면 agent는 계속 진행합니다. 대다수 공개 SaaS API(GitHub, Slack, Linear, Stripe, Notion, Salesforce, Asana...)는 이미 MCP server가 있거나 한나절이면 하나로 감쌀 수 있으며, 모두 좋은 MCP 후보입니다.

경험적인 기준은 다음과 같습니다. bearer token으로 공개 인터넷에서 접근할 수 있는 서비스라면 MCP toolset이 작동합니다. 자체 네트워크 안에서만 접근할 수 있다면 대신 custom tool을 사용하세요. gate notebook에서 이를 다룹니다.

**Vault**는 "token은 어디에 둘까요?"라는 질문의 답입니다. 세션 생성 시 token 하나를 hard-code하는 것은 단일 tenant 설정에서는 작동하지만 최종 사용자가 생기는 순간 무너집니다. 사용자마다 자신의 GitHub credential이 필요하고 서로 격리해야 합니다. vault는 사용자별 credential container로, 한 번 등록한 뒤 해당 사용자를 위해 생성하는 모든 세션에서 ID로 참조합니다. 자체 secret store를 운영하지 않고 요청마다 token을 전달하지 않으며, audit trail이 vault와 연결되어 agent가 어느 최종 사용자를 대신했는지 항상 알 수 있습니다.


## 1. 최종 사용자용 vault 생성하기

vault에는 Console에 표시되는 `display_name`과, 보통 내부 사용자 ID를 보관하는 `metadata` dict가 있으므로 자체 database의 레코드로 다시 mapping할 수 있습니다.


In [ ]:
vault = client.beta.vaults.create(
    display_name="Cookbook demo user",
    metadata={"internal_user_id": "u_demo_001", "team": "engineering"},
)
print(f"vault: {vault.id}")

## 2. MCP credential 연결하기

credential은 vault 아래에 있습니다. 각 credential은 MCP server URL과 agent가 그 server를 호출할 때 사용하는 token을 연결합니다. GitHub Copilot MCP server에는 정적 bearer token(GitHub PAT)이 가장 간단한 형태입니다. API는 필요한 서비스에 refresh를 포함한 전체 OAuth 흐름도 지원하며 두 형태 모두 `auth=`로 처리합니다.


In [ ]:
credential = client.beta.vaults.credentials.create(
    vault_id=vault.id,
    display_name="GitHub Copilot",
    auth={
        "type": "static_bearer",
        "mcp_server_url": "https://api.githubcopilot.com/mcp/",
        "token": GH_TOKEN,
    },
)
print(f"credential: {credential.id}")

## 3. 세션에서 vault 참조하기

`sessions.create`에 `vault_ids=[vault.id]`를 전달하면 API는 모든 tool 호출에서 일치하는 MCP server URL을 조회합니다. agent는 token 자체를 전혀 보지 않고 요청에 전달할 필요도 없습니다. agent 정의는 평소처럼 MCP server를 나열하기만 하며 credential 연결은 세션 생성 시 이루어집니다.


In [ ]:
agent = client.beta.agents.create(
    name="cookbook-operate",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-operate-in-production"},
    model=MODEL,
    system="You navigate GitHub repositories on behalf of the logged-in user.",
    mcp_servers=[
        {
            "type": "url",
            "name": "github",
            "url": "https://api.githubcopilot.com/mcp/",
        }
    ],
    tools=[
        {
            "type": "mcp_toolset",
            "mcp_server_name": "github",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

env = client.beta.environments.create(
    name="cookbook-operate-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    vault_ids=[vault.id],
    title="Operate demo",
)
print(f"session: {session.id}")

## 4. 최종 사용자로 turn 실행하기

이제 agent가 GitHub에서 하는 모든 작업은 vault의 credential을 통합니다. 자체 시스템의 audit은 간단합니다. 1단계에서 설정한 metadata로 vault가 사용자와 연결되어 있으므로 어느 최종 사용자의 작업인지 정확히 압니다.


In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "List the three most recently updated repos in the anthropics org.",
                }
            ],
        }
    ],
)
print("--- vault-backed MCP call ---")
stream_until_end_turn(client, session.id)

## 5. production HITL용 webhook

gate notebook의 streaming 패턴은 모든 것이 하나의 process에서 이루어져 개발 중 편리합니다. 하지만 사람이 검토하는 동안 HTTP 연결을 열어 두며, 규모를 늘리기 어렵고 process 재시작에도 유지되지 않습니다. 대신 production 패턴은 Console에 `session.status_idled`에서 발생하는 webhook을 등록합니다. 이는 agent가 완료되었거나 tool 결과를 기다리고 있다는 신호입니다.

webhook이 발생하면 server는 event를 검사하고 대기 중인 escalation을 검토자에게 제시하며, 사람이 완료할 때마다 `user.custom_tool_result`를 POST로 돌려보냅니다. 세션은 응답할 때까지 idle에 머물며 server 측에 장시간 연결이 없습니다.

세션에 `budget`이 설정되어 있다면([`CMA_cap_session_spend.ipynb`](CMA_cap_session_spend.ipynb) 참고) `session.budget_reached`도 등록하세요. 세션의 정가 비용이 상한에 도달할 때 한 번 발생하며, 감독 process가 세션별로 상한을 올릴지 작업을 일시 정지 상태로 둘지 결정하는 데 필요한 신호입니다. 세션 id가 포함되므로 handler는 idle handler와 같은 형태입니다. 세션을 조회하고, 결정하고, 실행합니다.

webhook 등록은 Console의 **Settings → Webhooks**에서 한 번 수행합니다. 생성 시 한 번만 표시되는 `whsec_...` 서명 secret을 받습니다. secrets manager에 저장하세요.

**아래 block은 notebook 셀이 아니라 reference 구현입니다.** 자체 server에 복사하세요. cookbook에서 설치하지 않는 FastAPI에 의존하며 이 notebook 흐름의 일부로 실행되지 않습니다. gate notebook의 agent 정의와 함께 사용하면 production server에서 gate workflow 전체를 진행하기에 충분합니다.

```python
import hmac
import hashlib
import json

from fastapi import FastAPI, Header, HTTPException, Request

app = FastAPI()
WEBHOOK_SECRET = "whsec_..."  # Console에서 받으며 secrets manager에서 불러오세요


def verify(body: bytes, sig: str) -> bool:
    expected = hmac.new(WEBHOOK_SECRET.encode(), body, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, sig)


@app.post("/webhooks/anthropic")
async def receive(req: Request, x_anthropic_signature: str = Header()):
    body = await req.body()
    if not verify(body, x_anthropic_signature):
        raise HTTPException(401)

    event = json.loads(body)
    session_id = event["resource_id"]

    if event["event_type"] == "session.status_idled":
        # agent가 idle이 되었습니다. 완료했거나 escalate()를 호출하여
        # user.custom_tool_result를 기다립니다. 최신 event를 보고
        # 판단하세요.
        events = client.beta.sessions.events.list(session_id=session_id)
        pending = [
            e for e in events.data
            if e.type == "agent.custom_tool_use" and e.name == "escalate"
            and not has_result(events, e.id)
        ]
        if pending:
            for tu in pending:
                enqueue_for_review(session_id, tu.id, tu.input)
        else:
            finalize_run(session_id)

    elif event["event_type"] == "session.budget_reached":
        # 작업을 유지한 채 지출 상한에서 세션이 일시 정지했습니다.
        # budget을 올려 계속 진행하거나 일시 정지 상태로 두고
        # 운영 담당자에게 알리세요.
        review_budget(session_id)

    return {"ok": True}


# 사람이 결정하면 review UI에서 호출합니다

def submit_review(session_id: str, custom_tool_use_id: str, decision: str):
    client.beta.sessions.events.send(
        session_id=session_id,
        events=[{
            "type": "user.custom_tool_result",
            "custom_tool_use_id": custom_tool_use_id,
            "content": [{"type": "text",
                         "text": json.dumps({"human_decision": decision})}],
        }],
    )
```

agent에 응답하는 코드는 gate notebook의 Part A 루프와 동일합니다. 바뀌는 것은 server가 할 일이 있음을 아는 방식뿐입니다. 로컬 루프가 event를 가져오는 대신 webhook이 일정에 맞춰 알림을 push합니다.


## 6. inference 지역 고정하기

compliance가 model 요청을 특정 지역에서 처리하도록 요구하면 workspace 기본값에 의존하지 말고 agent에 고정하세요.

```python
agent = client.beta.agents.create(
    name="residency-pinned agent",
    model={"id": MODEL, "inference_geo": "us"},
    ...
)
```

허용 값은 workspace의 `allowed_inference_geos`로 제한되는 `"global"`과 `"us"`입니다. 설정하지 않으면 각 요청은 처리 시점의 workspace `default_inference_geo`를 따르므로 나중에 기본값을 변경하면 실행 중인 세션에도 적용됩니다. 설정하면 agent 저장 시, 세션 생성 시, 모든 turn에서 고정값을 검증합니다. workspace allowlist를 좁히면 기존 설정을 예외로 인정하는 대신 이제 허용되지 않는 agent의 새 세션을 차단하고 실행 중인 세션의 다음 turn도 거부합니다. 이 엄격함 덕분에 지역 고정을 residency 제어로 사용할 수 있습니다. `sessions.create`의 `agent_with_overrides`는 공유 agent를 편집하지 않고 세션 하나를 다른 허용 지역으로 옮깁니다. 고정값을 해제하는 update 동작을 포함한 전체 설명은 [`CMA_pin_inference_geo.ipynb`](CMA_pin_inference_geo.ipynb)에 있습니다.


## 7. Resource lifecycle: list, retrieve, update, archive

API의 모든 resource, 즉 agent, environment, session, vault, credential은 같은 다섯 동작 패턴을 제공합니다. `list`, `retrieve`, `update`, `archive`, 그리고 일부에는 `delete`입니다. agent로 전체 동작을 보여 준 뒤 빠른 참고용으로 다른 각 resource의 사용 가능한 동작을 나열합니다.

**archive와 delete:** `archive`는 audit과 조회를 위해 레코드를 유지하지만 실행 중인 container를 종료하고 resource를 workspace quota 계산에서 제외합니다. `delete`는 레코드를 완전히 제거합니다. 대부분의 workflow에는 `archive`가 적합합니다. 레코드 제거가 명확히 필요할 때만(예: 테스트 정리) `delete`를 사용하세요.


In [ ]:
listed = client.beta.agents.list(limit=5)
print(f"workspace has at least {len(listed.data)} agents")

retrieved = client.beta.agents.retrieve(agent.id)
print(f"retrieved agent: name={retrieved.name} version={retrieved.version}")

# agent를 update하면 새 version이 생성됩니다. 현재
# version을 전달하여 알려진 상태에서 update하는지 확인하세요.
# 다른 process의 동시 update가 있으면 조용히 덮어쓰는 대신
# 해당 update가 거부됩니다.
updated = client.beta.agents.update(
    agent.id,
    version=agent.version,
    system="You navigate GitHub repositories. Be terse.",
)
print(f"updated to version {updated.version}")

# 모든 과거 version을 query할 수 있습니다.
versions = client.beta.agents.versions.list(agent_id=agent.id)
print(f"agent has {len(versions.data)} versions")

## 8. 정리하기

credential과 vault에는 자체 archive endpoint가 있습니다. vault를 archive해도 credential은 자동으로 archive되지 않으므로 완전히 정리하려면 credential부터 처리하세요.


In [ ]:
wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
client.beta.vaults.credentials.archive(credential.id, vault_id=vault.id)
client.beta.vaults.archive(vault.id)
print("archived")

## 다른 cookbook

이 notebook은 production 형태의 마무리입니다. 아직 실행하지 않았다면 이와 연결되는 workflow notebook부터 실행해 볼 만합니다.

- [`CMA_iterate_fix_failing_tests.ipynb`](CMA_iterate_fix_failing_tests.ipynb) — 시작점 notebook입니다. 실패하는 test suite의 실행-관찰-수정 루프로 agent, environment, session, 파일 mount, streaming event 루프를 소개합니다.
- [`CMA_orchestrate_issue_to_pr.ipynb`](CMA_orchestrate_issue_to_pr.ipynb) — 모의 gh CLI로 issue부터 merge된 PR까지 진행하며 체인 도중 CI 실패와 review comment에서 복구하는 multi-turn agent입니다.
- [`CMA_explore_unfamiliar_codebase.ipynb`](CMA_explore_unfamiliar_codebase.ipynb) — 오래된 문서 함정을 넣은 grounding 패턴입니다. 실행 중인 세션에 context를 추가하는 `sessions.resources.add`도 보여 줍니다.
- [`CMA_gate_human_in_the_loop.ipynb`](CMA_gate_human_in_the_loop.ipynb) — 위 webhook reference block과 함께 사용하는 human-in-the-loop workflow용 custom tool `decide()`와 `escalate()` 왕복 과정입니다.
